# STEP 5: Baselines and Four Model Families (Stage 1 Screening & Architecture)

This notebook documents the execution and evaluation of:
1. **Purged Walk-Forward Cross-Validation** (6 expanding folds on train post-warmup, $N=218,935$ rows, 12-candle purge, 15% internal early-stopping split).
2. **Benchmark Baselines Suite** (12 canonical baselines across 4 candidate horizons $N \in \{2, 3, 6, 12\}$).
3. **Stage 1 Screening** (76 Logistic Regression configurations on 70 `LINEAR_FEATURES`).
4. **Checkpoint 1 Selection** (Identification of top 3 survivors + mandatory fallback $N=12, S_1$, Setup B).

**Strict Data Partition Guard:** Evaluated strictly on the `train` partition. Validation rows remain reserved for Step 6, and test rows remain strictly locked.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

ML_DIR = Path.cwd()
if (ML_DIR / "ML").exists():
    ML_DIR = ML_DIR / "ML"
elif ML_DIR.name == "notebooks":
    ML_DIR = ML_DIR.parent
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

from src import config
from src.column_dictionary import load_master, get_feature_columns
from src.cv import PurgedWalkForwardCV

print(f"Working Directory: {ML_DIR}")
print(f"Candidate Horizons: {config.CANDIDATE_HORIZONS}")
print(f"Round-trip cost c: {config.TOTAL_ROUNDTRIP_COST:.4f}")

Working Directory: C:\B.TECH\Academics\SEMESTER V\03 Machine Learning\.ML LABs\Project\Stock-Market-ML-Indicator\ML
Candidate Horizons: [2, 3, 6, 12]
Round-trip cost c: 0.0023


## 1. Purged Walk-Forward Fold Boundaries

In [2]:
cv_folds_path = ML_DIR / "results" / "reports" / "cv_folds_summary.csv"
df_folds = pd.read_csv(cv_folds_path)

# Display fold structure
display_cols = [
    "fold", "train_rows", "purge_rows", "held_out_rows",
    "train_start_time", "train_end_time",
    "held_out_start_time", "held_out_end_time"
]
print(df_folds[display_cols].to_string(index=False))

total_held_out = df_folds["held_out_rows"].sum()
print(f"\nTotal evaluated held-out candles across 6 folds: {total_held_out:,}")
print(f"Purge gap strictly enforced between train and test: {df_folds['purge_rows'].iloc[0]} candles")

 fold  train_rows  purge_rows  held_out_rows          train_start_time            train_end_time       held_out_start_time         held_out_end_time
    1       86400          12          22088 2023-11-08 00:10:00+00:00 2024-09-03 00:05:00+00:00 2024-09-03 01:10:00+00:00 2024-11-18 17:45:00+00:00
    2      108488          12          22087 2023-11-08 00:10:00+00:00 2024-11-18 16:45:00+00:00 2024-11-18 17:50:00+00:00 2025-02-03 10:20:00+00:00
    3      130575          12          22087 2023-11-08 00:10:00+00:00 2025-02-03 09:20:00+00:00 2025-02-03 10:25:00+00:00 2025-04-21 02:55:00+00:00
    4      152662          12          22087 2023-11-08 00:10:00+00:00 2025-04-21 01:55:00+00:00 2025-04-21 03:00:00+00:00 2025-07-06 19:30:00+00:00
    5      174749          12          22087 2023-11-08 00:10:00+00:00 2025-07-06 18:30:00+00:00 2025-07-06 19:35:00+00:00 2025-09-21 12:05:00+00:00
    6      196836          12          22087 2023-11-08 00:10:00+00:00 2025-09-21 11:05:00+00:00 2025-09-2

## 2. Benchmark Baselines Evaluation (Part 3)

12 canonical baselines evaluated on identical out-of-fold predictions across all 4 candidate horizons ($N \in \{2, 3, 6, 12\}$).

In [3]:
baselines_path = ML_DIR / "results" / "reports" / "baselines_oof_summary.csv"
df_baselines = pd.read_csv(baselines_path)

# Summary table sorted by break-even gap
b_cols = [
    "baseline", "horizon", "signals_per_day", "hit_rate",
    "break_even_rate", "break_even_gap", "net_return_bps", "mcc"
]
top_baselines = df_baselines.sort_values(by="break_even_gap", ascending=False).reset_index(drop=True)
print("Top 10 Baselines by Break-Even Gap:")
print(top_baselines.head(10)[b_cols].to_string(index=False))

# Key comparisons
best_b = top_baselines.iloc[0]
print(f"\nBest Baseline: {best_b['baseline']} at N={best_b['horizon']} (Gap: {best_b['break_even_gap']:.4f}, Net: {best_b['net_return_bps']:.2f} bps)")

Top 10 Baselines by Break-Even Gap:
            baseline  horizon  signals_per_day  hit_rate  break_even_rate  break_even_gap  net_return_bps       mcc
    loc_position_288       12       181.603448  0.514591         0.837433       -0.322841      -22.787260  0.032015
reverse_rule_vote_4b       12        69.691970  0.516095         0.839704       -0.323609      -23.252639  0.033051
              rsi_14       12       181.607795  0.511012         0.837440       -0.326428      -22.052582  0.000000
          ema50_rule       12       181.607795  0.511012         0.837440       -0.326428      -22.052582  0.000000
    reverse_momentum       12       181.307811  0.510047         0.837282       -0.327235      -22.414246  0.020140
            momentum       12       181.307811  0.489953         0.837282       -0.347329      -23.585754 -0.020140
      reverse_rsi_14       12       181.607795  0.488988         0.837440       -0.348452      -23.947418  0.000000
     ref_5feature_lr       12       

## 3. Stage 1 Screening: 76 Logistic Regression Configurations (Part 4)

Budget: 76 configurations evaluated on `FS_CORE` `LINEAR_FEATURES` (70 features):
- **Setup B** (Binary raw sign): 4 horizons.
- **Setup A** (Clear move): 4 horizons $\times$ 9 distinct rules = 36 configurations.
- **Setup C** (Three-class): 4 horizons $\times$ 9 distinct rules = 36 configurations.

Common ranking metric on top 20% most confident signals: $\text{rank\_metric} = \mu_{\text{gap}} - \sigma_{\text{gap}}$.
Minimum volume filter: $\ge 15$ signals/day.

In [4]:
stage1_path = ML_DIR / "results" / "reports" / "stage1_lr_ranking.csv"
df_stage1 = pd.read_csv(stage1_path)

s_cols = [
    "config_id", "setup", "horizon", "rule", "signals_per_day",
    "mean_hit_rate", "mean_break_even_gap", "std_break_even_gap",
    "rank_metric", "mean_net_bps", "status"
]

survivors = df_stage1[df_stage1["status"].isin(["survivor", "fallback"])].copy()
print("=== STAGE 1 SURVIVORS & FALLBACK CONFIGURATION ===")
print(survivors[s_cols].to_string(index=False))

print("\n=== TOP 10 CANDIDATE CONFIGURATIONS ===")
print(df_stage1.head(10)[s_cols].to_string(index=False))

rejected_cnt = (df_stage1["status"] == "rejected").sum()
print(f"\nConfigurations rejected by volume filter (< 15 sig/day): {rejected_cnt} / {len(df_stage1)}")

=== STAGE 1 SURVIVORS & FALLBACK CONFIGURATION ===
config_id   setup  horizon     rule  signals_per_day  mean_hit_rate  mean_break_even_gap  std_break_even_gap  rank_metric  mean_net_bps   status
 C_12_F15 setup_c       12      F15        28.782010       0.528970            -0.175858            0.038144    -0.214002    -21.269923 survivor
 C_12_F10 setup_c       12      F10        48.050375       0.535283            -0.198893            0.040747    -0.239639    -21.003567 survivor
 C_12_F05 setup_c       12      F05        58.017930       0.538663            -0.225862            0.051633    -0.277495    -21.083109 survivor
     B_12 setup_b       12 raw_sign        66.626857       0.546112            -0.291696            0.084368    -0.376064    -21.119889 fallback

=== TOP 10 CANDIDATE CONFIGURATIONS ===
config_id   setup  horizon     rule  signals_per_day  mean_hit_rate  mean_break_even_gap  std_break_even_gap  rank_metric  mean_net_bps    status
 C_12_F15 setup_c       12      F15  

## 4. Checkpoint 1 Baseline vs. Model Comparison

In [5]:
# Comparison between top Stage 1 model and top baseline
top_model = df_stage1[df_stage1["status"] == "survivor"].iloc[0]
top_base = df_baselines.sort_values(by="break_even_gap", ascending=False).iloc[0]

print(f"Top Model:    {top_model['config_id']} (Setup {top_model['setup']}, N={top_model['horizon']}, Rule {top_model['rule']})")
print(f"  Break-Even Gap: {top_model['mean_break_even_gap']:.4f} (std: {top_model['std_break_even_gap']:.4f})")
print(f"  Rank Metric:    {top_model['rank_metric']:.4f}")
print(f"  Net Return:     {top_model['mean_net_bps']:.2f} bps/trade")
print(f"  Signals/Day:    {top_model['signals_per_day']:.1f}")

print(f"\nTop Baseline: {top_base['baseline']} (N={top_base['horizon']})")
print(f"  Break-Even Gap: {top_base['break_even_gap']:.4f}")
print(f"  Net Return:     {top_base['net_return_bps']:.2f} bps/trade")
print(f"  Signals/Day:    {top_base['signals_per_day']:.1f}")

Top Model:    C_12_F15 (Setup setup_c, N=12, Rule F15)
  Break-Even Gap: -0.1759 (std: 0.0381)
  Rank Metric:    -0.2140
  Net Return:     -21.27 bps/trade
  Signals/Day:    28.8

Top Baseline: loc_position_288 (N=12)
  Break-Even Gap: -0.3228
  Net Return:     -22.79 bps/trade
  Signals/Day:    181.6
